### Structured Tool Inputs with Pydantic

In [1]:
import os
import logging
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import AgentExecutor, create_tool_calling_agent
from pydantic import BaseModel, Field

In [2]:
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
logging.getLogger('httpx').setLevel(logging.WARNING)

In [3]:
load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

First Multi-Argument Tool

In [4]:
# Step 1: Define the schema — what inputs this tool expects
class CropSearchInput(BaseModel):
    disease: str = Field(description='The disease name, e.g., Cassava Mosaic, Maize Smut')
    crop: str = Field(description='The crop affected, e.g., cassava, maize, rice')
    language: str = Field(description='Language for the answer: English, Yoruba, Hausa, or Igbo')
    
# Step 2: Define the tool using that schema
@tool(args_schema=CropSearchInput)
def search_crop_disease(disease: str, crop: str, language: str) -> str:
    '''
    Search the crop disease knowledge base and return the answer in the requested language.
    '''
    
    knowledge = {
        'cassava mosaic': 'Use disease-free cuttings, plant resistant varieties, remove infected plants.',
        'maize smut':     'Plant resistant hybrids, rotate crops, remove galls before they burst.',
        'rice blast':     'Use resistant varieties, balance nitrogen, apply fungicides when needed.',
    }
    
    # Find a matching disease
    for key, control in knowledge.items():
        if key.split()[0] in disease.lower() and key.split()[0] in crop.lower():
            
            # Fake translation for demonstration
            prefix = f'[{language}] ' if language.lower() != 'english' else ''
            return f'{prefix}{disease.title()} on {crop}: {control}'
        
    return f'No knowledge found for {disease} on {crop}'

print('search_crop_disease tool ready')
print(f'Type: {type(search_crop_disease).__name__}')

search_crop_disease tool ready
Type: StructuredTool


Test

In [5]:
result = search_crop_disease.invoke({
    'disease': 'cassava mosaic',
    'crop': 'cassava',
    'language': 'Yoruba'
})

print(result)

[Yoruba] Cassava Mosaic on cassava: Use disease-free cuttings, plant resistant varieties, remove infected plants.


A Tool With a Numeric Argument

In [6]:
# Step 1: Schema
class WeatherInput(BaseModel):
    city: str = Field(description='The city name, e.g., Lagos, Abuja, Kano')
    days: int = Field(description='Number of forecast days, from 1 to 7')
    
# Step 2: Tool definition
@tool(args_schema=WeatherInput)
def get_forecast(city: str, days: int) -> str:
    '''
    Return a multi-day weather forecast for a city.
    '''
    
    if days < 1:
        return 'Error: days must be at least 1.'
    
    if days > 7:
        return 'Error: I can only forecast up to 7 days ahead.'
    
    fake = {
        'lagos': ['Sunny 32°C', 'Cloudy 30°C', 'Rain 28°C', 'Sunny 31°C',
                  'Cloudy 30°C', 'Sunny 33°C', 'Rain 29°C'],
        'abuja': ['Cloudy 28°C', 'Sunny 30°C', 'Cloudy 27°C', 'Sunny 29°C',
                  'Cloudy 28°C', 'Sunny 31°C', 'Cloudy 28°C'],
    }
    
    forecast = fake.get(city.lower(), None)
    if forecast is None:
        return f'No forecast data for {city}'
    
    lines = [f'Day {i+1}: {forecast[i]}' for i in range(days)]
    return f'{days}-day forecast for {city.title()}:\n' + '\n'.join(lines)

print('get_forecast tool ready')
print(f'   Type: {type(get_forecast).__name__}')

get_forecast tool ready
   Type: StructuredTool


In [7]:
print(get_forecast.invoke({'city': 'Lagos', 'days': 5}))

5-day forecast for Lagos:
Day 1: Sunny 32°C
Day 2: Cloudy 30°C
Day 3: Rain 28°C
Day 4: Sunny 31°C
Day 5: Cloudy 30°C


In [8]:
print(get_forecast.invoke({'city': 'Abuja', 'days': 2}))
print()
print(get_forecast.invoke({'city': 'Lagos', 'days': 10}))   # over the limit

2-day forecast for Abuja:
Day 1: Cloudy 28°C
Day 2: Sunny 30°C

Error: I can only forecast up to 7 days ahead.


A Tool That Can Fail

In [9]:
# Step 1: Schema
class FarmerLookupInput(BaseModel):
    farmer_id: str = Field(description='The farmer ID, e.g., F001, F002, F003')
    
# Step 2: Tool with fake records and error handling
@tool(args_schema=FarmerLookupInput)
def get_farmer_record(farmer_id: str) -> str:
    '''
    Look up a farmer's profile and their registered crops.
    '''
    
    farmers = {
        'F001': 'Adebayo Okafor — crops: cassava, maize — location: Oyo',
        'F002': 'Fatima Ibrahim — crops: rice, tomato — location: Kano',
        'F003': 'Chinedu Eze — crops: cassava, yam — location: Enugu',
    }
    
    record = farmers.get(farmer_id.upper())
    if record is None:
        return f'No farmer found with ID: {farmer_id}. Valid IDs look like F001, F002, F003.'
    
    return record

print('get_farmer_record tool ready')
print(f'   Type: {type(get_farmer_record).__name__}')

get_farmer_record tool ready
   Type: StructuredTool


In [10]:
print(get_farmer_record.invoke({'farmer_id': 'F001'}))
print()
print(get_farmer_record.invoke({'farmer_id': 'F002'}))
print()
print(get_farmer_record.invoke({'farmer_id': 'F999'})) 

Adebayo Okafor — crops: cassava, maize — location: Oyo

Fatima Ibrahim — crops: rice, tomato — location: Kano

No farmer found with ID: F999. Valid IDs look like F001, F002, F003.


Collect the Tools Into a List

In [11]:
tools = [search_crop_disease, get_forecast, get_farmer_record]

print(f'{len(tools)} tools ready for the agent:\n')

for t in tools:
    print(f'    - {t.name}')
    print(f'    {t.description[:80]}...')
    print()

3 tools ready for the agent:

    - search_crop_disease
    Search the crop disease knowledge base and return the answer in the requested la...

    - get_forecast
    Return a multi-day weather forecast for a city....

    - get_farmer_record
    Look up a farmer's profile and their registered crops....



Inspect the Schema of One Tool

In [12]:
print('Tool name:', search_crop_disease.name)
print()
print(search_crop_disease.description)
print()
print('Input schema:')
print(search_crop_disease.args_schema.schema())

Tool name: search_crop_disease

Search the crop disease knowledge base and return the answer in the requested language.

Input schema:
{'properties': {'disease': {'description': 'The disease name, e.g., Cassava Mosaic, Maize Smut', 'title': 'Disease', 'type': 'string'}, 'crop': {'description': 'The crop affected, e.g., cassava, maize, rice', 'title': 'Crop', 'type': 'string'}, 'language': {'description': 'Language for the answer: English, Yoruba, Hausa, or Igbo', 'title': 'Language', 'type': 'string'}}, 'required': ['disease', 'crop', 'language'], 'title': 'CropSearchInput', 'type': 'object'}


Build the Agent's Prompt

In [13]:
prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a helpful farming assistant. '
     'Use the available tools when they help answer the user\'s question. '
     'Fill in every tool argument carefully based on the user\'s question. '
     'If the user does not specify a value, choose a sensible default (e.g., English for language).'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}'),
])

print('Agent prompt ready')

Agent prompt ready


Create the Agent and Executor

In [14]:
agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

print('Agent ready')

Agent ready


Simple test

In [15]:
agent_executor.invoke({'input': 'Search for cassava mosaic disease on cassava in English.'})



> Entering new AgentExecutor chain...

Invoking: `search_crop_disease` with `{'disease': 'Cassava Mosaic', 'crop': 'cassava', 'language': 'English'}`


Cassava Mosaic on cassava: Use disease-free cuttings, plant resistant varieties, remove infected plants.Cassava Mosaic on cassava can be managed by using disease-free cuttings, planting resistant varieties, and removing infected plants.

> Finished chain.


{'input': 'Search for cassava mosaic disease on cassava in English.',
 'output': 'Cassava Mosaic on cassava can be managed by using disease-free cuttings, planting resistant varieties, and removing infected plants.'}

In [16]:
agent_executor.invoke({'input': 'Search for cassava mosaic disease on cassava in Pidgin.'})



> Entering new AgentExecutor chain...

Invoking: `search_crop_disease` with `{'disease': 'Cassava Mosaic', 'crop': 'cassava', 'language': 'English'}`


Cassava Mosaic on cassava: Use disease-free cuttings, plant resistant varieties, remove infected plants.Here be di information for Cassava Mosaic wey affect cassava:

- Use cuttings wey no get disease.
- Plant varieties wey dey resistant.
- Remove plants wey don catch di disease.

> Finished chain.


{'input': 'Search for cassava mosaic disease on cassava in Pidgin.',
 'output': 'Here be di information for Cassava Mosaic wey affect cassava:\n\n- Use cuttings wey no get disease.\n- Plant varieties wey dey resistant.\n- Remove plants wey don catch di disease.'}

In [17]:
agent_executor.invoke({'input': 'Search for cassava mosaic disease on cassava in Yoruba.'})



> Entering new AgentExecutor chain...

Invoking: `search_crop_disease` with `{'disease': 'Cassava Mosaic', 'crop': 'cassava', 'language': 'Yoruba'}`


[Yoruba] Cassava Mosaic on cassava: Use disease-free cuttings, plant resistant varieties, remove infected plants.Here is the information on Cassava Mosaic disease affecting cassava in Yoruba:

**Cassava Mosaic**: Lo awọn gige ti ko ni arun, gbin awọn irugbin ti o ni resistance, yọ awọn ọgbin ti o ni arun.

> Finished chain.


{'input': 'Search for cassava mosaic disease on cassava in Yoruba.',
 'output': 'Here is the information on Cassava Mosaic disease affecting cassava in Yoruba:\n\n**Cassava Mosaic**: Lo awọn gige ti ko ni arun, gbin awọn irugbin ti o ni resistance, yọ awọn ọgbin ti o ni arun.'}

In [18]:
agent_executor.invoke({'input': 'Search for cassava mosaic disease on cassava in Hausa.'})



> Entering new AgentExecutor chain...

Invoking: `search_crop_disease` with `{'disease': 'Cassava Mosaic', 'crop': 'cassava', 'language': 'Hausa'}`


[Hausa] Cassava Mosaic on cassava: Use disease-free cuttings, plant resistant varieties, remove infected plants.Ga bayani akan cutar Cassava Mosaic a kan cassava a cikin Hausa: Yi amfani da katako mara cuta, shuka nau'ikan da suka yi juriya, kuma cire shukar da ta kamu da cutar.

> Finished chain.


{'input': 'Search for cassava mosaic disease on cassava in Hausa.',
 'output': "Ga bayani akan cutar Cassava Mosaic a kan cassava a cikin Hausa: Yi amfani da katako mara cuta, shuka nau'ikan da suka yi juriya, kuma cire shukar da ta kamu da cutar."}

In [19]:
agent_executor.invoke({'input': 'Search for cassava mosaic disease on cassava in Igbo.'})



> Entering new AgentExecutor chain...

Invoking: `search_crop_disease` with `{'disease': 'Cassava Mosaic', 'crop': 'cassava', 'language': 'Igbo'}`


[Igbo] Cassava Mosaic on cassava: Use disease-free cuttings, plant resistant varieties, remove infected plants.For cassava mosaic disease on cassava, the recommendations in Igbo are: 

- Jiri mkpụrụ osisi na-enweghị ọrịa,
- Tinye ụdị ndị na-eguzogide ọrịa,
- Wepụ osisi ndị nwere ọrịa.

> Finished chain.


{'input': 'Search for cassava mosaic disease on cassava in Igbo.',
 'output': 'For cassava mosaic disease on cassava, the recommendations in Igbo are: \n\n- Jiri mkpụrụ osisi na-enweghị ọrịa,\n- Tinye ụdị ndị na-eguzogide ọrịa,\n- Wepụ osisi ndị nwere ọrịa.'}

Test 2 — Missing arguments

In [20]:
agent_executor.invoke({'input': 'Search for maize smut.'})



> Entering new AgentExecutor chain...

Invoking: `search_crop_disease` with `{'disease': 'Maize Smut', 'crop': 'maize', 'language': 'English'}`


Maize Smut on maize: Plant resistant hybrids, rotate crops, remove galls before they burst.Maize Smut is a disease that affects maize. To manage this disease, you can take the following actions:

- Plant resistant hybrids.
- Rotate crops.
- Remove galls before they burst. 

These practices can help mitigate the impact of Maize Smut on your maize crops.

> Finished chain.


{'input': 'Search for maize smut.',
 'output': 'Maize Smut is a disease that affects maize. To manage this disease, you can take the following actions:\n\n- Plant resistant hybrids.\n- Rotate crops.\n- Remove galls before they burst. \n\nThese practices can help mitigate the impact of Maize Smut on your maize crops.'}

In [25]:
agent_executor.invoke({'input': 'Search for maize smut in Yoruba.'})



> Entering new AgentExecutor chain...

Invoking: `search_crop_disease` with `{'disease': 'Maize Smut', 'crop': 'maize', 'language': 'Yoruba'}`


[Yoruba] Maize Smut on maize: Plant resistant hybrids, rotate crops, remove galls before they burst.In Yoruba, the information about Maize Smut on maize is as follows: 

"Maize Smut lori maize: Gba awọn irugbin ti o ni resistance, yipada awọn irugbin, yọ awọn galls ṣaaju ki wọn to fọ."

> Finished chain.


{'input': 'Search for maize smut in Yoruba.',
 'output': 'In Yoruba, the information about Maize Smut on maize is as follows: \n\n"Maize Smut lori maize: Gba awọn irugbin ti o ni resistance, yipada awọn irugbin, yọ awọn galls ṣaaju ki wọn to fọ."'}

Test Multi-Tool

In [24]:
agent_executor.invoke({'input': 'What is the 2-day forecast for Kano, and look up farmer F002?'})



> Entering new AgentExecutor chain...



Invoking: `get_forecast` with `{'city': 'Kano', 'days': 2}`


No forecast data for Kano
Invoking: `get_farmer_record` with `{'farmer_id': 'F002'}`


Fatima Ibrahim — crops: rice, tomato — location: KanoThere is no forecast data available for Kano at this time. 

As for farmer F002, their details are as follows:
- **Name:** Fatima Ibrahim
- **Crops:** Rice, Tomato
- **Location:** Kano

> Finished chain.


{'input': 'What is the 2-day forecast for Kano, and look up farmer F002?',
 'output': 'There is no forecast data available for Kano at this time. \n\nAs for farmer F002, their details are as follows:\n- **Name:** Fatima Ibrahim\n- **Crops:** Rice, Tomato\n- **Location:** Kano'}